# Detecting Local Models: Ollama & Hugging Face

This notebook demonstrates how to discover models installed locally. We
inspect:

* Ollama (via its CLI)
* the Hugging Face cache directory

The examples below show how to list those models and also how one might
load a Hugging Face model for inference afterwards.

In [1]:
# Install and import libraries
# !pip install transformers huggingface-hub

import subprocess
import shutil
from pathlib import Path

## Ollama Model Detection

In [2]:
def ollama_models() -> list[str]:
    """Return a list of model tags currently installed in Ollama."""
    try:
        proc = subprocess.run(["ollama", "list"], capture_output=True, text=True, check=True)
    except FileNotFoundError:
        raise RuntimeError("Ollama CLI not found; install from https://ollama.com")
    models = [line.strip() for line in proc.stdout.splitlines() if line.strip()]
    return models

models = ollama_models()
print("Ollama models installed:")
if models:
    for m in models:
        print("  -", m)
else:
    print("  (none found)")


Ollama models installed:
  - NAME                ID              SIZE      MODIFIED
  - llama3.2:1b         baf6a787fdff    1.3 GB    4 weeks ago
  - deepseek-r1:1.5b    e0979632db5a    1.1 GB    4 weeks ago
  - llama3.2:latest     a80c4f17acd5    2.0 GB    4 weeks ago
  - smollm:1.7b         95f6557a0f0f    990 MB    4 weeks ago
  - llama3.2:3b         a80c4f17acd5    2.0 GB    4 weeks ago
  - falcon3:3b          784a7eab5b89    2.0 GB    4 weeks ago
  - gemma3:1b           8648f39daa8f    815 MB    4 weeks ago
  - qwen2.5:3b          357c53fb659c    1.9 GB    4 weeks ago
  - phi3:3.8b           4f2222927938    2.2 GB    5 weeks ago


## Hugging Face Cached Detection

In [3]:
def format_size(bytes_size: int) -> str:
    """Format bytes to human readable size."""
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if bytes_size < 1024:
            return f"{bytes_size:.2f} {unit}"
        bytes_size /= 1024
    return f"{bytes_size:.2f} TB"

def hf_cached_models() -> list[tuple[str, int]]:
    """Return list of (name, size_bytes) for directories under the Hugging Face hub cache.

    This corresponds to models you have previously downloaded via
    `from_pretrained` or `hf_hub_download`.
    """
    cache_dir = Path.home() / ".cache" / "huggingface" / "hub"
    if not cache_dir.exists():
        return []
    
    def get_dir_size(path: Path) -> int:
        total = 0
        for p in path.rglob('*'):
            if p.is_file():
                total += p.stat().st_size
        return total
    
    models = []
    for p in cache_dir.iterdir():
        if p.is_dir():
            size = get_dir_size(p)
            models.append((p.name, size))
    return models

models = hf_cached_models()
print("Hugging Face cached models:")
if models:
    for name, size in models:
        print(f"  - {name}: {format_size(size)}")
else:
    print("  (none found)")

Hugging Face cached models:
  (none found)


## Delete Cached Hugging Face Models

In [4]:
# delete cached hugging face models
cache_dir = Path.home() / ".cache" / "huggingface" / "hub"

if cache_dir.exists():
    for child in cache_dir.iterdir():
        if child.is_dir():
            print(f"removing {child}")
            shutil.rmtree(child)
else:
    print("no hugging face cache directory found")